# Rayleigh-Benard Convection: Discovering Nu(Ra, Pr) and the Aspect-Ratio Degeneracy

This notebook is a port of `final_ex_sir.ipynb` to a 3-parameter Rayleigh-Benard (RB) problem.

**Parameters** (all in log10):
- $X_1 = \log_{10} \mathrm{Ra}$, $X_2 = \log_{10} \mathrm{Pr}$, $X_3 = \log_{10} \Gamma$

**Data:** windowed time series of Nusselt and Reynolds numbers (Grossmann-Lohse-style surrogate; swap for Dedalus DNS in production).

**Expected discoveries:**
- $\eta_0$ is a power law in $\mathrm{Ra}$, $\mathrm{Pr}$ tracking $\log \mathrm{Nu}$.
- $\eta_1$ is a different linear combination of $\log \mathrm{Ra}$, $\log \mathrm{Pr}$ tracking $\log \mathrm{Re}$.
- $\eta_2 \approx \log \Gamma$: aspect ratio is the *degenerate* direction (Nu, Re depend on it only weakly for $\Gamma \gtrsim 0.5$).

## Pipeline
1. RB simulator (Grossmann-Lohse surrogate)
2. Generate train/test data
3. **`MinMaxScaler((1, 2))` on theta** -- scaler kept around so we can map SR expressions back to physical theta after step 6
4. Fisher-network ensemble
5. Normalising-flow flattening
6. Coordinate alignment + symbolic regression + postprocessing
7. Map expressions back to physical theta and compare with Grossmann-Lohse

In [ ]:
# Colab setup (skip if running locally with the package already installed)
!git clone https://github.com/tlmakinen/degeneracy_distillery.git
%cd /content/degeneracy_distillery
!pip install -q -e .
%cd /content/
!git clone https://github.com/DeaglanBartlett/ESR.git
%cd /content/ESR
!pip install -q -e .

# RESTART your runtime after the install cell above

In [ ]:
import os
# os.kill(os.getpid(), 9)  # uncomment in Colab to restart

In [ ]:
import esr.generation.generator  # sanity check
import numpy as np
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
import jax.random as jr
from scipy.optimize import brentq
from tqdm import tqdm
import sympy
import degeneracy_distillery
from degeneracy_distillery.training_loop_fishnets import train_fishnets

plt.rcParams.update({
    'font.size': 12, 'axes.labelsize': 14, 'axes.titlesize': 14,
    'xtick.labelsize': 11, 'ytick.labelsize': 11, 'legend.fontsize': 10,
    'figure.figsize': (8, 5), 'figure.dpi': 130,
    'savefig.dpi': 200, 'savefig.bbox': 'tight',
})

## 1. Simulator (Grossmann-Lohse surrogate)

We solve the simplified GL coupled equations for $(\mathrm{Nu}, \mathrm{Re})(\mathrm{Ra}, \mathrm{Pr})$ and add a small linear correction in $\Gamma$ (so it shows up as a *weakly* identifiable direction, mostly degenerate). Replace `simulate_rb` with a Dedalus call to upgrade to real DNS.

In [ ]:
# --- Grossmann-Lohse implicit closure (textbook constants from Stevens 2013) ---
_C3, _C4, _A_GL = 0.487, 0.0252, 0.922
_C1, _C2 = 8.05, 1.38

def _nu_from_re(Re, Pr):
    num = _C3 * Re ** 0.5 * Pr ** 0.5
    den = max(1.0 - _C4 * Re ** 0.25 * Pr ** 0.25, 1e-3)
    return 1.0 + num / den

def _gl_residual_re(Re, Ra, Pr):
    Nu = _nu_from_re(Re, Pr)
    eps_u = (Nu - 1.0) * Ra * Pr ** -2
    f_eps = _C1 * Re ** -1 + _C2 * Re ** -0.25
    return eps_u - f_eps * Re ** 3

def grossmann_lohse(Ra, Pr):
    Re = brentq(_gl_residual_re, 1.0, 1e8, args=(Ra, Pr), xtol=1e-3)
    return float(_nu_from_re(Re, Pr)), float(Re)

In [ ]:
# Box priors on (log10 Ra, log10 Pr, log10 Gamma) -- analogue of (beta, gamma, I0/10)
LOG_RA_MIN, LOG_RA_MAX = 6.0, 12.0
LOG_PR_MIN, LOG_PR_MAX = -1.0, 2.0     # Pr in [0.1, 100]
LOG_G_MIN, LOG_G_MAX = -0.3, 0.6        # Gamma in [0.5, 4]

n_windows = 30
noise_frac_nu = 0.05
noise_frac_re = 0.07

def simulate_rb(log_ra, log_pr, log_g, rng):
    """Forward model: returns (Nu(t_1..t_N), Re(t_1..t_N)) -- shape (2*n_windows,).

    Drop-in DNS swap: replace this body with a Dedalus 2-D RB run that returns
    n_windows turnover-time-averaged Nu(t), Re(t) values.
    """
    Ra, Pr, Gamma = 10.0 ** log_ra, 10.0 ** log_pr, 10.0 ** log_g
    Nu_m, Re_m = grossmann_lohse(Ra, Pr)
    aspect = 1.0 + 0.02 * (Gamma - 1.0)         # very weak Gamma effect
    Nu_m *= aspect
    Re_m *= aspect ** 0.5
    nu_t = Nu_m * (1.0 + noise_frac_nu * rng.standard_normal(n_windows))
    re_t = Re_m * (1.0 + noise_frac_re * rng.standard_normal(n_windows))
    return np.concatenate([nu_t, re_t]).astype(np.float32)

def generate_rb(n, rng, desc):
    log_ra = rng.uniform(LOG_RA_MIN, LOG_RA_MAX, n)
    log_pr = rng.uniform(LOG_PR_MIN, LOG_PR_MAX, n)
    log_g  = rng.uniform(LOG_G_MIN,  LOG_G_MAX,  n)
    theta = np.stack([log_ra, log_pr, log_g], axis=1).astype(np.float32)
    data = np.array([simulate_rb(*row, rng) for row in tqdm(theta, desc=desc)],
                    dtype=np.float32)
    # log-transform Nu, Re channels (their dynamic range spans many decades)
    return theta, np.log10(np.maximum(data, 1e-6))

In [ ]:
nsims = 5000
rng_train = np.random.default_rng(42)
rng_test = np.random.default_rng(43)

theta_train, data_train = generate_rb(nsims, rng_train, "Training sims")
theta_test,  data_test  = generate_rb(nsims, rng_test,  "Test sims")

print(f"theta_train: {theta_train.shape}, data_train: {data_train.shape}")
print(f"theta box (log10): Ra in {[LOG_RA_MIN, LOG_RA_MAX]}, "
      f"Pr in {[LOG_PR_MIN, LOG_PR_MAX]}, Gamma in {[LOG_G_MIN, LOG_G_MAX]}")

## 2. Pre-fishnet rescaling

The dynamic ranges across $\log_{10} \mathrm{Ra}, \log_{10} \mathrm{Pr}, \log_{10} \Gamma$ differ by an order of magnitude (6 vs 3 vs 0.9). MinMax-scaling theta to $[1, 2]^3$ removes the scale mismatch before fishnet training and stabilises both flattening and SR. We keep the `scaler` object around so we can:
1. Push SR-input batches through the same affine map,
2. Symbolically convert the discovered expressions back to physical theta after SR.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler((1, 2))
scaler.fit(theta_train)
theta_train_s = scaler.transform(theta_train).astype(np.float32)
theta_test_s  = scaler.transform(theta_test).astype(np.float32)

print("data_min_:", scaler.data_min_)
print("data_max_:", scaler.data_max_)
print("scaled theta range:", theta_train_s.min(0), theta_train_s.max(0))

np.savez(
    "rb_data_scaled.npz",
    theta_train=theta_train_s, data_train=data_train,
    theta_test=theta_test_s,   data_test=data_test,
    maxtheta=scaler.data_max_, mintheta=scaler.data_min_,
)

In [ ]:
# Quick visualisation of the simulator outputs
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for idx in rng_train.choice(nsims, 5, replace=False):
    axes[0].plot(data_train[idx, :n_windows], alpha=0.7,
                 label=f"logRa={theta_train[idx,0]:.1f}")
axes[0].set(xlabel="window", ylabel=r"$\log_{10}\mathrm{Nu}(t)$",
            title="Example Nu traces")
axes[0].legend(fontsize=8)

log_nu_mean = data_train[:, :n_windows].mean(1)
sc = axes[1].scatter(theta_train[:, 0], theta_train[:, 1], c=log_nu_mean,
                     s=2, alpha=0.6, cmap="viridis")
plt.colorbar(sc, ax=axes[1], label=r"$\langle\log_{10}\mathrm{Nu}\rangle$")
axes[1].set(xlabel=r"$\log_{10}\mathrm{Ra}$", ylabel=r"$\log_{10}\mathrm{Pr}$",
            title="Parameter space")

axes[2].hist(log_nu_mean, bins=50, alpha=0.7, density=True)
axes[2].set(xlabel=r"$\langle\log_{10}\mathrm{Nu}\rangle$", ylabel="density",
            title="Nu distribution")
plt.tight_layout()
plt.savefig("rb_data_overview.pdf")
plt.show()

## 3. Fisher network ensemble

All three theta components go into the network -- unlike SIR there's no scalar nuisance to peel off.

In [ ]:
import flax.linen as nn

embedding_net = nn.Sequential([
    nn.Dense(64), nn.gelu,
    nn.Dense(32), nn.gelu,
])

_ = train_fishnets(
    theta_train_s, data_train,
    theta_test_s, data_test,
    num_models=10,
    train_epochs=5000,
    patience=30,
    n_layers=[2, 5],
    hids_min=50, hids_max=300,
    embedding_net=embedding_net,
    lr=5e-5,
    train_batch_size=200,
    outdir="fishnets-log-rb",
)

## 4. Flattening normalising flow

In [ ]:
from degeneracy_distillery.training_loop_flatten import fit_flattening

fish_npz = np.load("fishnets-log-rb/fishnets_outputs.npz")
thetas = jnp.array(fish_npz["theta"])
ensemble_weights = fish_npz["ensemble_weights"]
F_network_ensemble = jnp.array(fish_npz["Fs"])
print("thetas", thetas.shape, "F", F_network_ensemble.shape)

shared_kwargs = dict(
    flattener_activation="softplus",
    loss_type="log_frob",
    ensemble_weights=ensemble_weights,
    forward_backward_mlp=True,
    forward_backward_invertibility_weight=1.0,
    n_layers=8,
    offset=0.0,
    beta_det=0.1,
    noise=1e-2,
    batch_size=250,
    finetune_epochs=200,
    epochs_phase1=1000,
    epochs_phase2=500,
    lr_phase1=2e-6,
    lr_schedule_initial=7e-5,
    lr_decay=0.3,
    l1_alpha=0.0,
    do_plot=False,
)
w, ensemble_ws, output_dict, flatten_model = fit_flattening(
    θs=thetas,
    F_network_ensemble=F_network_ensemble,
    output_prefix="rb_grossmann_lohse",
    Fisher_to_flatten="average",
    return_model=True,
    **shared_kwargs,
)

In [ ]:
# Cheap SR-only theta pool, scaled with the same affine map
rng_sr = np.random.default_rng(456)
n_sr = 10_000
theta_sr_phys = np.stack([
    rng_sr.uniform(LOG_RA_MIN, LOG_RA_MAX, n_sr),
    rng_sr.uniform(LOG_PR_MIN, LOG_PR_MAX, n_sr),
    rng_sr.uniform(LOG_G_MIN,  LOG_G_MAX,  n_sr),
], axis=1).astype(np.float32)
X_sr = scaler.transform(theta_sr_phys).astype(np.float32)

ys_sr = jnp.array([
    jax.vmap(lambda x: flatten_model.apply(w_i, x))(X_sr)
    for w_i in ensemble_ws
])
print("X_sr", X_sr.shape, "ys_sr", ys_sr.shape)

## 5. Coordinate alignment (`align_coords.load_and_process_data_v2`)

In [ ]:
from degeneracy_distillery.align_coords import load_and_process_data_v2
from degeneracy_distillery.preprocessing_utils import weighted_std

data = load_and_process_data_v2(
    datapath="./",
    filename="rb_grossmann_lohse.npz",
    num_samps=4000,
    seed=44,
    process_ensemble=True,
    n_d=1.0,
    align_mode="kabsch",
    separate_nonlinearity=False,
    canonicalize="sign_only",
    use_prior_normalization=True,
    restore_reference_mean=False,
    Fisher_to_flatten="average",
    verbose=False,
)

X = data["X"]
mask = X[:, 0] > 0.0
X = X[mask]
y = data["y"][mask]
ys = np.array([yy[mask] for yy in data["ys"]])
y_std = data["y_std"][mask]
dy_sr = data["dy_sr"][mask]
Fs = data["Fs"][mask]
n_params = X.shape[1]

# Pull the floor up so y >= 1 (PyOperon prefers strictly-positive inputs)
ymin_ = (y.min(0) - 1.0)
ys -= ymin_
y -= ymin_

# Apply the same rotation/centering to the SR pool
ys_sr_rot = np.array([
    np.einsum("ij,bj->bi", data["rotmats"][i], ys_sr[i] - ys_sr[i].mean(0))
    for i in range(len(ys_sr))
])
y_std_sr = weighted_std(ys_sr_rot, data["ensemble_weights"])
y_sr = np.average(ys_sr_rot, 0, data["ensemble_weights"])
ys_sr_rot -= y_sr.min(0)
y_sr -= y_sr.min(0)
y_sr += 1
ys_sr_rot += 1
print("X", X.shape, "y", y.shape, "X_sr", X_sr.shape, "y_sr", y_sr.shape)

In [ ]:
# X_i vs y_j sanity grid in the flattened frame
def plot_X_vs_y(X, y, y_std, skip=10):
    nx, ny = X.shape[1], y.shape[1]
    fig, axs = plt.subplots(nx, ny, figsize=(2.5 * ny, 2.0 * nx), squeeze=False)
    sub = slice(None, None, skip)
    for i in range(nx):
        for j in range(ny):
            ax = axs[i, j]
            ax.errorbar(X[sub, i], y[sub, j], yerr=y_std[sub, j],
                        fmt="none", ecolor="0.7", elinewidth=0.5, zorder=1)
            ax.scatter(X[sub, i], y[sub, j], s=6, alpha=0.4,
                       c=X[sub, (i + 1) % nx], cmap="viridis", zorder=2)
            ax.set(xlabel=fr"$X_{i}$", ylabel=fr"$y_{j}$")
            ax.tick_params(labelsize=8)
    plt.tight_layout()
    plt.show()

plot_X_vs_y(X, y, y_std, skip=10)

## 6. Symbolic regression

We follow the SIR convention of feeding `X + 1` (keeps inputs strictly positive after rotation/centering) to PyOperon.

In [ ]:
from degeneracy_distillery.sr_utils import fit_and_analyze_sr

mdl_coords, frob_coords, analysis, split_data = fit_and_analyze_sr(
    X + 1, y, y_std, dy_sr, Fs,
    n_params=n_params,
    parent_dir="./sr_results_rb/",
    test_size=0.5,
    random_state=32134,
    shuffle=True,
    time_limit=60 * 5,
    max_length=25,
    max_depth=10,
    allowed_symbols="add,mul,div,pow,constant,variable,sqrt",
    max_complexity_thresh=20,
    equation_set="pareto",
)
print("MDL  coords:", mdl_coords)
print("Frob coords:", frob_coords)

In [ ]:
from degeneracy_distillery.sr_utils import analyze_equations, sr_structure_predicate
from degeneracy_distillery.postprocessing_utils import print_discovered_expressions

mdl_coords, frob_coords, analysis = analyze_equations(
    X + 1, y, y_std, dy_sr, Fs,
    parent_dir="sr_results_rb/",
    n_params=n_params,
    equation_set="pareto",
    max_complexity_thresh=20,
    length_penalty=2.0,
    equation_predicate=sr_structure_predicate(
        n_params=n_params,
        forbid_self_transcendental=False,
        check_nested_exp=False,
    ),
)
print_discovered_expressions([sympy.simplify(p).evalf(2) for p in mdl_coords])

## 7. Postprocessing (`postprocess_new`)

In [ ]:
from degeneracy_distillery.postprocess_new import (
    analyze_atom_sharing, regroup_like_terms,
)

report = analyze_atom_sharing(mdl_coords)
pruned_exprs, R, info = regroup_like_terms(
    mdl_coords, X=X, Fs=Fs, n_params=n_params,
    method="atoms",
    do_snap=True, snap_rel_tol=0.1, snap_flat_tol=0.1,
    decimal=2,
    threshold=1.0,
)
print_discovered_expressions([sympy.simplify(p).evalf(2) for p in pruned_exprs])

In [ ]:
from degeneracy_distillery.sr_utils import check_symbolic_invertibility

inv = check_symbolic_invertibility(pruned_exprs, verbose=True)
print("\nInverse coords (eta -> theta):", inv["inv_coords"])

## 8. Map expressions back to physical theta

Symbolic regression sees `X = scaler.transform(theta_phys)`, i.e. 

$$X_i = a_i\,\theta_i + b_i, \quad a_i = \mathrm{scale}\_[i],\ b_i = \mathrm{min}\_[i].$$

Substituting this into each expression gives a closed form in physical $\theta$. We package this as a single helper so the user can flip between scaled and physical views at will.

In [ ]:
def to_physical_theta(exprs, scaler):
    """Substitute X_i = a_i * theta_i + b_i into each sympy expression."""
    a = scaler.scale_
    b = scaler.min_
    n = len(a)
    X_syms     = sympy.symbols(" ".join(f"X{i+1}" for i in range(n)))
    theta_syms = sympy.symbols(" ".join(f"theta{i+1}" for i in range(n)))
    subs = {X_syms[i]: float(a[i]) * theta_syms[i] + float(b[i]) for i in range(n)}
    return [sympy.simplify(sympy.sympify(e).subs(subs)) for e in exprs]

physical_exprs = to_physical_theta(pruned_exprs, scaler)
print("Discovered eta in physical (log10 Ra, log10 Pr, log10 Gamma):")
for k, e in enumerate(physical_exprs):
    print(f"  eta_{k} = {e.evalf(3)}")

## 9. Compare with Grossmann-Lohse asymptotics

Two sanity checks:
1. **Flatness** of the network Fisher in raw vs ad-hoc vs learned coordinates.
2. **Correlation** of the learned $\eta$ with $\log\mathrm{Nu}$, $\log\mathrm{Re}$ and $\log\Gamma$ -- we expect $\eta_2$ to track $\log\Gamma$ (the degenerate direction).

In [ ]:
from degeneracy_distillery.postprocessing_utils import (
    flatten_with_numerical_jacobian, check_flattening, get_y_sr,
)
from scipy.stats import pearsonr

X_test = X + 1
Fs_test = Fs
dy_test = dy_sr

nn_flats = jax.vmap(flatten_with_numerical_jacobian)(dy_test, Fs_test)
mdl_flats, _ = check_flattening(mdl_coords, X=X_test, Fs=Fs_test)
pruned_flats, _ = check_flattening(pruned_exprs, X=X_test, Fs=Fs_test)

# Ad-hoc Grossmann-Lohse-inspired coords (in scaled X, just for comparison)
adhoc_coords = ["X1", "X2", "X3"]
adhoc_flats, _ = check_flattening(adhoc_coords, X=X_test, Fs=Fs_test)

def flat_score(Q):
    return np.linalg.norm(Q - np.eye(Q.shape[-1]), axis=(-2, -1))

for name, Q in [("raw theta", Fs_test),
                ("adhoc (X1,X2,X3)", adhoc_flats),
                ("MDL", mdl_flats),
                ("pruned", pruned_flats),
                ("NN", nn_flats)]:
    print(f"{name:18s}  median ||Q-I||_F = {np.median(flat_score(np.asarray(Q))):.3f}")

In [ ]:
# Did we recover the aspect-ratio degenerate direction?
y_eval = get_y_sr(pruned_exprs, jnp.array(X_test))
phys_test = scaler.inverse_transform(X_test - 1)  # back to (logRa, logPr, logG)
log_ra, log_pr, log_g = phys_test.T

physics = {
    r"$\log_{10}\mathrm{Ra}$": log_ra,
    r"$\log_{10}\mathrm{Pr}$": log_pr,
    r"$\log_{10}\Gamma$": log_g,
}
fig, axes = plt.subplots(n_params, len(physics),
                         figsize=(3.5 * len(physics), 3.0 * n_params),
                         squeeze=False)
for j, (lab, p) in enumerate(physics.items()):
    for i in range(n_params):
        r, _ = pearsonr(np.asarray(y_eval[:, i]), p)
        ax = axes[i, j]
        ax.scatter(p[::4], y_eval[::4, i], s=2, alpha=0.2)
        ax.set(xlabel=lab, ylabel=fr"$\eta_{i}$",
               title=f"r = {r:+.2f}")
fig.suptitle("Learned eta vs physical control parameters", y=1.02)
plt.tight_layout()
plt.savefig("rb_eta_vs_physics.pdf")
plt.show()

In [ ]:
# Save artefacts
import pickle
with open("sr_results_rb/sr_expressions.pkl", "wb") as f:
    pickle.dump({
        "mdl_coords": mdl_coords,
        "frob_coords": frob_coords,
        "pruned_exprs": pruned_exprs,
        "physical_exprs": [str(e) for e in physical_exprs],
        "inv_coords": inv["inv_coords"],
        "scaler_scale": scaler.scale_,
        "scaler_min":   scaler.min_,
        "scaler_data_min": scaler.data_min_,
        "scaler_data_max": scaler.data_max_,
    }, f)
print("saved sr_results_rb/sr_expressions.pkl")